# Draft a Unified Table for **Points of Interest** in Berlin Layers

## Step 1: Gather info on tables already in the database

### Import libraries

In [2]:
# Import Libraries
import osmnx as ox # to fetch data from OpenStreetMap
import geopandas as gpd # to work with geospatial data
import pandas as pd
from sqlalchemy import create_engine, text
import warnings
import json

warnings.filterwarnings("ignore")

### Credentials

In [ ]:
user_name=''
password=''

### Create the connection

In [4]:
# Conection
host = 'localhost'
port = '5433'
database = 'layereddb'
schema='berlin_source_data'

#connection to db after you opened tunnel
engine = create_engine(f'postgresql+psycopg2://{user_name}:{password}@{host}:{port}/{database}')

### Queries

- Show tables list that are only statitics

In [5]:
query = f"""
SELECT table_name
FROM information_schema.tables
WHERE 
    (table_schema = 'berlin_source_data' AND table_name LIKE '%stat%')
    OR 
    (table_schema = 'berlin_source_data' AND table_name LIKE '%agg%')
    OR 
    (table_schema = 'berlin_source_data' AND table_name LIKE '%price%')
ORDER BY table_name;
"""

# Execute the query
stats_tables_df = pd.read_sql(text(query), engine)
print(stats_tables_df)

                    table_name
0             crime_statistics
1    district_level_aggregated
2           districts_pop_stat
3                  land_prices
4          regional_statistics
5  rent_stats_per_neighborhood


In [6]:
# Loop through tables and show top 5 rows of stats tables
for table in stats_tables_df['table_name']:
    print(f"\n--- {table} ---")
    preview_query = f"SELECT * FROM berlin_source_data.{table} LIMIT 5;"
    df = pd.read_sql(text(preview_query), engine)
    display(df)


--- crime_statistics ---


,id,area_id,locality,district,district_id,year,crime_type_german,crime_type_english,category,total_number_cases,frequency_100k,population_base,severity_weight,created_at,updated_at
0,2001,13007.0,Osloer Straße,Mitte,11001001,2015,Brand- stiftung,Arson,Property Crime,13,34.0,None,4.5,2025-08-21 12:52:44.976746,2025-08-21 12:52:44.976746
1,2002,13008.0,Brunnenstraße Nord,Mitte,11001001,2015,Brand- stiftung,Arson,Property Crime,10,27.0,None,4.5,2025-08-21 12:52:44.976746,2025-08-21 12:52:44.976746
2,2003,14009.0,Parkviertel,Mitte,11001001,2015,Brand- stiftung,Arson,Property Crime,16,36.0,None,4.5,2025-08-21 12:52:44.976746,2025-08-21 12:52:44.976746
3,2004,14010.0,Wedding Zentrum,Mitte,11001001,2015,Brand- stiftung,Arson,Property Crime,14,25.0,None,4.5,2025-08-21 12:52:44.976746,2025-08-21 12:52:44.976746
4,2005,19900.0,"Bezirk (Mi), nicht zuzuordnen",Mitte,11001001,2015,Brand- stiftung,Arson,Property Crime,8,NaN,None,4.5,2025-08-21 12:52:44.976746,2025-08-21 12:52:44.976746



--- district_level_aggregated ---


,district_id,district,long_term_listings_count,avg_long_term_listings_price,median_long_term_listings_price,avg_long_term_listings_rooms,avg_long_term_listings_surface_sqm,rooms_1_count,rooms_1_5_count,rooms_2_count,...,latest_land_value_others_per_sqm,latest_land_value_residential_area_per_sqm,latest_mean_net_rent_sqm,protection_zones_count,total_area_em_ha,total_area_es_ha,dental_offices_count,dental_wheelchair_yes_count,dental_wheelchair_no_count,dental_wheelchair_limited_count
0,11001001,Mitte,175,2045.90,1750.0,2.33,79.35,39,3,59,...,9000.0,1600.0,19.91,32.0,686.6,658.4,82,18,16,8
1,11002002,Friedrichshain-Kreuzberg,105,1770.07,1714.0,2.39,69.51,16,1,43,...,15.0,3000.0,19.26,17.0,981.3,604.5,57,11,20,2
2,11003003,Pankow,154,1869.21,1724.5,2.78,81.77,21,3,47,...,10.0,2000.0,17.65,22.0,833.2,458.2,93,8,22,12
3,11004004,Charlottenburg-Wilmersdorf,130,1987.59,1612.5,2.58,80.97,24,0,44,...,10.0,1800.0,19.40,18.0,397.2,237.2,136,9,17,9
4,11005005,Spandau,104,1551.15,1584.5,3.07,90.78,6,3,31,...,40.0,40.0,13.32,3.0,205.4,57.7,16,1,4,0



--- districts_pop_stat ---


...

,district,standard_land_value_per_sqm,typical_land_use_type,typical_floor_space_ratio,land_use_category,district_id,year
0,Friedrichshain-Kreuzberg,400.0,W - Wohngebiet,0.025,Residential Area,11002002,2012
1,Reinickendorf,550.0,M2 - Mischgebiet,0.020,Mixed Area,11012012,2012
2,Friedrichshain-Kreuzberg,650.0,M2 - Mischgebiet,0.030,Mixed Area,11002002,2012
3,Tempelhof-Schöneberg,1300.0,M2 - Mischgebiet,0.030,Mixed Area,11007007,2012
4,Spandau,180.0,G - Gewerbe,NaN,Commercial Area,11005005,2012



--- regional_statistics ---


,district_id,district,year,inhabitants,total_area_ha,share_forest_water_agriculture,forest_area_ha,water_area_ha,agriculture_area_ha,population_density_per_ha,number_of_residences,living_space_per_resident_m2
0,11004004,Charlottenburg-Wilmersdorf,2012,298567,6472,0.295,1622,281,794,46.1,184637,48.9
1,11002002,Friedrichshain-Kreuzberg,2012,259483,2034,0.067,4,132,1,127.6,145328,39.3
2,11011011,Lichtenberg,2012,258586,5212,0.140,51,104,293,49.6,143895,37.0
3,11010010,Marzahn-Hellersdorf,2012,248786,6178,0.060,173,117,573,40.3,127149,37.1
4,11001001,Mitte,2012,329969,3947,0.036,0,142,0,83.6,185209,39.2



--- rent_stats_per_neighborhood ---


,district_id,district,median_net_rent_per_m2,number_of_cases,mean_net_rent_per_m2,year
0,11001001,Mitte,19.91,4169,19.91,2024
1,11002002,Friedrichshain-Kreuzberg,19.42,2509,19.26,2024
2,11003003,Pankow,17.00,3636,17.65,2024
3,11004004,Charlottenburg-Wilmersdorf,19.39,3036,19.40,2024
4,11005005,Spandau,12.00,2403,13.32,2024


### Table name and columns

In [7]:
# Loop through tables and show top 5 rows of stats tables
for table in stats_tables_df['table_name']:
    print(f"\n--- {table} ---")
    preview_query = f"SELECT column_name, data_type FROM information_schema.columns WHERE table_schema = 'berlin_source_data' AND table_name = '{table}';"
    cols_df = pd.read_sql(text(preview_query), engine)
    # display(cols_df)
    print(list(cols_df.itertuples(index=False, name=None)))


--- crime_statistics ---
[('id', 'integer'), ('area_id', 'character varying'), ('locality', 'character varying'), ('district', 'character varying'), ('district_id', 'character varying'), ('year', 'smallint'), ('crime_type_german', 'character varying'), ('crime_type_english', 'character varying'), ('category', 'character varying'), ('total_number_cases', 'integer'), ('frequency_100k', 'double precision'), ('population_base', 'integer'), ('severity_weight', 'double precision'), ('created_at', 'timestamp without time zone'), ('updated_at', 'timestamp without time zone')]

--- district_level_aggregated ---
[('district_id', 'bigint'), ('district', 'text'), ('long_term_listings_count', 'bigint'), ('avg_long_term_listings_price', 'double precision'), ('median_long_term_listings_price', 'double precision'), ('avg_long_term_listings_rooms', 'double precision'), ('avg_long_term_listings_surface_sqm', 'double precision'), ('rooms_1_count', 'bigint'), ('rooms_1_5_count', 'bigint'), ('rooms_2_coun

### Aggregated each table to District level as some were in row level (exactly one row per district per year), where a table didnt have a year it just inserts the total it has so all years show the same

In [8]:
query = """
    WITH rent_agg AS (
        SELECT district_id, year, AVG(mean_net_rent_per_m2) AS avg_rent
        FROM berlin_source_data.rent_stats_per_neighborhood
        GROUP BY district_id, year
    ),
    land_agg AS (
        SELECT district_id, year, AVG(standard_land_value_per_sqm) AS avg_land_value
        FROM berlin_source_data.land_prices
        GROUP BY district_id, year
    )
    SELECT 
        r.district_id,
        r.district,
        r.year,
        r.inhabitants AS population_total,
        p.male AS male_population,
        p.female AS female_population,
        p.germans AS german_population,
        p.foreigners AS foreigners_population,
        p.single AS single_population,
        p.married AS married_population,
        p.divorced AS divorced_population,
        a.violent_crime_cases_count AS violent_crimes,
        a.crime_weighted_score AS crime_score,
        r.total_area_ha,
        r.living_space_per_resident_m2,
        l.avg_land_value,
        rent.avg_rent
    FROM berlin_source_data.regional_statistics r
    LEFT JOIN berlin_source_data.districts_pop_stat p 
        ON r.district_id::TEXT = p.district_id::TEXT
    LEFT JOIN berlin_source_data.district_level_aggregated a
        ON r.district_id::TEXT = a.district_id::TEXT
    LEFT JOIN land_agg l
        ON r.district_id::TEXT = l.district_id::TEXT 
    AND r.year::TEXT = l.year::TEXT
    LEFT JOIN rent_agg rent
        ON r.district_id::TEXT = rent.district_id::TEXT 
    AND r.year::TEXT = rent.year::TEXT
    ORDER BY district_id, year
    ;
"""
# Execute the query
district_overview_df = pd.read_sql(text(query), engine)
district_overview_df

,district_id,district,year,population_total,male_population,female_population,german_population,foreigners_population,single_population,married_population,divorced_population,violent_crimes,crime_score,total_area_ha,living_space_per_resident_m2,avg_land_value,avg_rent
0,11001001,Mitte,2012,329969,204542,192462,249053,147951,240196,113277,28623,0,45.0,3947,39.2,2092.967033,8.48
1,11001001,Mitte,2013,337593,204542,192462,249053,147951,240196,113277,28623,0,45.0,3947,37.7,2123.913043,9.52
2,11001001,Mitte,2014,347336,204542,192462,249053,147951,240196,113277,28623,0,45.0,3947,37.0,2241.956522,9.80
3,11001001,Mitte,2015,354300,204542,192462,249053,147951,240196,113277,28623,0,45.0,3947,36.5,2821.758242,10.94
4,11001001,Mitte,2016,361986,204542,192462,249053,147951,240196,113277,28623,0,45.0,3947,36.9,3297.934783,10.94
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
139,11012012,Reinickendorf,2019,266408,135775,138323,208099,65999,131052,100221,24812,202,606.0,8932,39.4,419.200000,9.75
140,11012012,Reinickendorf,2020,266123,135775,138323,208099,65999,131052,100221,24812,202,606.0,8932,39.5,512.525253,9.29
141,11012012,Reinickendorf,2021,265772,135775,138323,208099,65999,131052,100221,24812,202,606.0,8932,39.0,531.929293,9.82
142,11012012,Reinickendorf,2022,268308,135775,138323,208099,65999,131052,100221,24812,202,606.0,8932,38.6,375.693396,10.53


### Full table - including breakdown by neighbourhood and land use type

In [9]:
query = """
SELECT r.district_id, 
    r.district, 
    r.year, 
    r.inhabitants AS population_total, 
    p.male AS male_population, p.female AS 
    female_population, 
    p.germans AS german_population, 
    p.foreigners AS foreigners_population, 
    p.single AS single_population, 
    p.married AS married_population, 
    p.divorced AS divorced_population, 
    a.violent_crime_cases_count AS violent_crimes, 
    a.crime_weighted_score AS crime_score, 
    r.total_area_ha, 
    r.living_space_per_resident_m2, 
    l.standard_land_value_per_sqm AS avg_land_value, 
    rent.mean_net_rent_per_m2 AS avg_rent 
FROM berlin_source_data.regional_statistics r 
LEFT JOIN berlin_source_data.districts_pop_stat p 
    ON r.district_id::TEXT = p.district_id::TEXT 
LEFT JOIN berlin_source_data.district_level_aggregated a 
    ON r.district_id::TEXT = a.district_id::TEXT 
LEFT JOIN berlin_source_data.land_prices l 
    ON r.district_id::TEXT = l.district_id::TEXT 
    AND r.year::TEXT = l.year::TEXT 
LEFT JOIN berlin_source_data.rent_stats_per_neighborhood rent 
    ON r.district_id::TEXT = rent.district_id::TEXT 
    AND r.year::TEXT = rent.year::TEXT;
"""
# Execute the query
district_overview_full_df = pd.read_sql(text(query), engine)
district_overview_full_df.head()

,district_id,district,year,population_total,male_population,female_population,german_population,foreigners_population,single_population,married_population,divorced_population,violent_crimes,crime_score,total_area_ha,living_space_per_resident_m2,avg_land_value,avg_rent
0,11002002,Friedrichshain-Kreuzberg,2012,259483,150002,142622,202632,89992,188392,76076,19227,0,945.0,2034,39.3,400.0,8.56
1,11012012,Reinickendorf,2012,243239,135775,138323,208099,65999,131052,100221,24812,202,606.0,8933,41.7,550.0,6.55
2,11002002,Friedrichshain-Kreuzberg,2012,259483,150002,142622,202632,89992,188392,76076,19227,0,945.0,2034,39.3,650.0,8.56
3,11007007,Tempelhof-Schöneberg,2012,320917,175519,181440,273377,83582,183234,121863,30838,0,4.0,5310,42.4,1300.0,7.58
4,11005005,Spandau,2012,218935,127361,131916,189914,69363,125965,93781,23333,0,7.0,9187,40.2,180.0,6.00


### create the district_details table

In [10]:
from sqlalchemy import text

query = """

DROP TABLE IF EXISTS district_details CASCADE;

CREATE TABLE IF NOT EXISTS district_details (
    district_id BIGINT NOT NULL,

    room_distribution_json JSONB,
    school_stats_json JSONB,
    pool_stats_json JSONB,

    crime_totals_json JSONB,
    crime_violent_json JSONB,
    crime_property_json JSONB,
    crime_drugs_json JSONB,
    crime_other_json JSONB,

    demographics_json JSONB,
    healthcare_json JSONB,
    transport_json JSONB,
    parks_recreation_json JSONB,
    land_use_json JSONB,
    ownership_json JSONB,
    real_estate_value_json JSONB,

    long_term_listings_json JSONB,
    short_term_listings_json JSONB,

    other_details_json JSONB
);

"""

# Execute the DDL
with engine.begin() as conn:   # begin() handles commit automatically
    conn.execute(text(query))

### Insert the rows into the table

In [11]:
insert_query = """
INSERT INTO district_details (
    district_id,

    room_distribution_json,
    school_stats_json,
    pool_stats_json,

    crime_totals_json,
    crime_violent_json,
    crime_property_json,
    crime_drugs_json,
    crime_other_json,

    demographics_json,
    healthcare_json,
    transport_json,
    parks_recreation_json,
    land_use_json,
    ownership_json,
    real_estate_value_json,

    long_term_listings_json,
    short_term_listings_json,

    other_details_json
)
SELECT
    dla.district_id,

    -- ROOM DISTRIBUTION JSON (counts + avg prices + avg surface)
    jsonb_build_object(
        'counts', jsonb_build_object(
            'rooms_1', rooms_1_count,
            'rooms_1_5', rooms_1_5_count,
            'rooms_2', rooms_2_count,
            'rooms_2_5', rooms_2_5_count,
            'rooms_3', rooms_3_count,
            'rooms_3_5', rooms_3_5_count,
            'rooms_4', rooms_4_count,
            'rooms_4_5', rooms_4_5_count,
            'rooms_5', rooms_5_count,
            'rooms_5_5', rooms_5_5_count,
            'rooms_6', rooms_6_count,
            'rooms_7', rooms_7_count,
            'rooms_7_5', rooms_7_5_count,
            'rooms_8', rooms_8_count,
            'rooms_9', rooms_9_count
        ),
        'avg_price', jsonb_build_object(
            '1', avg_price_rooms_1,
            '1_5', avg_price_rooms_1_5,
            '2', avg_price_rooms_2,
            '2_5', avg_price_rooms_2_5,
            '3', avg_price_rooms_3,
            '3_5', avg_price_rooms_3_5,
            '4', avg_price_rooms_4,
            '4_5', avg_price_rooms_4_5,
            '5', avg_price_rooms_5,
            '5_5', avg_price_rooms_5_5,
            '6', avg_price_rooms_6,
            '7', avg_price_rooms_7,
            '7_5', avg_price_rooms_7_5,
            '8', avg_price_rooms_8,
            '9', avg_price_rooms_9
        ),
        'avg_surface', jsonb_build_object(
            '1', avg_surface_rooms_1,
            '1_5', avg_surface_rooms_1_5,
            '2', avg_surface_rooms_2,
            '2_5', avg_surface_rooms_2_5,
            '3', avg_surface_rooms_3,
            '3_5', avg_surface_rooms_3_5,
            '4', avg_surface_rooms_4,
            '4_5', avg_surface_rooms_4_5,
            '5', avg_surface_rooms_5,
            '5_5', avg_surface_rooms_5_5,
            '6', avg_surface_rooms_6,
            '7', avg_surface_rooms_7,
            '7_5', avg_surface_rooms_7_5,
            '8', avg_surface_rooms_8,
            '9', avg_surface_rooms_9
        )
    ) AS room_distribution_json,

    -- SCHOOL STATS JSON (includes university_students_ratio_per_inhabitant)
    jsonb_build_object(
        'schools_count', schools_count,
        'students_count', students_count,
        'teachers_count', teachers_count,
        'primary_schools_count', primary_schools_count,
        'academic_secondary_schools_count', academic_secondary_schools_count,
        'integrated_secondary_schools_count', integrated_secondary_schools_count,
        'private_schools_count', private_schools_count,
        'special_needs_schools_count', special_needs_schools_count,
        'vocational_schools_count', vocational_schools_count,
        'other_schools_count', other_schools_count,
        'none_schools_count', none_schools_count,
        'students_per_teacher_ratio', students_per_teacher_ratio,
        'kindergartens_count', kindergartens_count,
        'university_students_ratio_per_inhabitant', university_students_ratio_per_inhabitant
    ) AS school_stats_json,

    -- POOL STATS JSON
    jsonb_build_object(
        'pools_count', pools_count,
        'freibad_count', freibad_count,
        'freizeitbad_count', freizeitbad_count,
        'hallenbad_count', hallenbad_count,
        'hotelbad_count', hotelbad_count,
        'klinikbad_count', klinikbad_count,
        'kombibad_count', kombibad_count,
        'naturbad_count', naturbad_count,
        'natuerliche_badestelle_count', natuerliche_badestelle_count,
        'schulbad_count', schulbad_count,
        'sonstiges_bad_count', sonstiges_bad_count,
        'open_all_year_pools_count', open_all_year_pools_count
    ) AS pool_stats_json,

    -- CRIME: Totals JSON (overall figures)
    jsonb_build_object(
        'crime_cases_total_count', crime_cases_total_count,
        'crime_weighted_score', crime_weighted_score,
        'avg_crime_frequency_100k', avg_crime_frequency_100k,
        'total_crimes_cases_count', total_crimes_cases_count,
        'overall_crime_cases_count', overall_crime_cases_count
    ) AS crime_totals_json,

    -- CRIME: Violent crimes JSON
    jsonb_build_object(
        'assault_total_cases_count', assault_total_cases_count,
        'serious_assault_cases_count', serious_assault_cases_count,
        'violent_crime_cases_count', violent_crime_cases_count,
        'robbery_cases_count', robbery_cases_count,
        'street_robbery_cases_count', street_robbery_cases_count
    ) AS crime_violent_json,

    -- CRIME: Property crimes JSON
    jsonb_build_object(
        'theft_total_cases_count', theft_total_cases_count,
        'theft_from_vehicles_cases_count', theft_from_vehicles_cases_count,
        'vehicle_theft_cases_count', vehicle_theft_cases_count,
        'residential_burglary_cases_count', residential_burglary_cases_count,
        'property_crime_cases_count', property_crime_cases_count,
        'property_damage_total_cases_count', property_damage_total_cases_count
    ) AS crime_property_json,

    -- CRIME: Drug-related JSON
    jsonb_build_object(
        'drug_crimes_cases_count', drug_crimes_cases_count,
        'drug_offense_cases_count', drug_offense_cases_count
    ) AS crime_drugs_json,

    -- CRIME: Other / misc JSON
    jsonb_build_object(
        'arson_cases_count', arson_cases_count,
        'arson_total_cases_count', arson_total_cases_count,
        'bicycle_theft_cases_count', bicycle_theft_cases_count,
        'coercion_and_threats_cases_count', coercion_and_threats_cases_count,
        'graffiti_vandalism_cases_count', graffiti_vandalism_cases_count,
        'neighborhood_crimes_cases_count', neighborhood_crimes_cases_count,
        'public_order_cases_count', public_order_cases_count
    ) AS crime_other_json,

    -- DEMOGRAPHICS JSON
    jsonb_build_object(
        'male_gender_population_count', male_gender_population_count,
        'female_gender_population_count', female_gender_population_count,
        'german_nationality_population_count', german_nationality_population_count,
        'foreign_nationality_population_count', foreign_nationality_population_count,
        'single_marital_population_count', single_marital_population_count,
        'married_marital_population_count', married_marital_population_count,
        'widowed_marital_population_count', widowed_marital_population_count,
        'divorced_marital_population_count', divorced_marital_population_count,
        'civil_partnership_marital_population_count', civil_partnership_marital_population_count,
        'evangelical_religion_population_count', evangelical_religion_population_count,
        'catholic_religion_population_count', catholic_religion_population_count,
        'other_or_none_religion_population_count', other_or_none_religion_population_count,
        'age_0_6_population_count', age_0_6_population_count,
        'age_6_15_population_count', age_6_15_population_count,
        'age_15_18_population_count', age_15_18_population_count,
        'age_18_27_population_count', age_18_27_population_count,
        'age_27_45_population_count', age_27_45_population_count,
        'age_45_55_population_count', age_45_55_population_count,
        'age_55_65_population_count', age_55_65_population_count,
        'age_65_plus_population_count', age_65_plus_population_count,
        'foreigners_population_pct', foreigners_population_pct,
        'germans_population_pct', germans_population_pct,
        'single_marital_population_pct', single_marital_population_pct,
        'children_0_6_population_pct', children_0_6_population_pct,
        'latest_population_count', latest_population_count
    ) AS demographics_json,

    -- HEALTHCARE JSON
    jsonb_build_object(
        'hospitals_count', hospitals_count,
        'hospital_beds_count', hospital_beds_count,
        'hospital_cases_count', hospital_cases_count,
        'hospital_beds_per_10000_inhabitants', hospital_beds_per_10000_inhabitants,
        'dental_offices_count', dental_offices_count,
        'dental_wheelchair_yes_count', dental_wheelchair_yes_count,
        'dental_wheelchair_no_count', dental_wheelchair_no_count,
        'dental_wheelchair_limited_count', dental_wheelchair_limited_count
    ) AS healthcare_json,

    -- TRANSPORT JSON
    jsonb_build_object(
        'transport_stop_density_per_ha', transport_stop_density_per_ha,
        'ubahn_stations_count', ubahn_stations_count,
        'ubahn_lines_count', ubahn_lines_count,
        'bus_tram_stops_count', bus_tram_stops_count
    ) AS transport_json,

    -- PARKS & RECREATION JSON
    jsonb_build_object(
        'parks_count', parks_count,
        'total_parks_area_sqm', total_parks_area_sqm,
        'playgrounds_count', playgrounds_count,
        'total_play_area_sqm', total_play_area_sqm,
        'venues_count', venues_count,
        'restaurants_count', restaurants_count,
        'bars_count', bars_count,
        'cafes_count', cafes_count
    ) AS parks_recreation_json,

    -- LAND USE JSON
    jsonb_build_object(
        'latest_total_area_ha', latest_total_area_ha,
        'latest_land_use_share_forest_water_agriculture', latest_land_use_share_forest_water_agriculture,
        'latest_forest_area_ha', latest_forest_area_ha,
        'latest_water_area_ha', latest_water_area_ha,
        'latest_agriculture_area_ha', latest_agriculture_area_ha,
        'latest_population_density_per_ha', latest_population_density_per_ha,
        'latest_residences_count', latest_residences_count,
        'latest_living_space_per_resident_sqm', latest_living_space_per_resident_sqm
    ) AS land_use_json,

    -- OWNERSHIP JSON
    jsonb_build_object(
        'private_ownership_count', private_ownership_count,
        'public_ownership_count', public_ownership_count,
        'no_ownership_count', no_ownership_count
    ) AS ownership_json,

    -- REAL ESTATE VALUE JSON
    jsonb_build_object(
        'latest_land_value_sqm', latest_land_value_sqm,
        'latest_land_value_commercial_area_per_sqm', latest_land_value_commercial_area_per_sqm,
        'latest_land_value_forest_agricultural_area_per_sqm', latest_land_value_forest_agricultural_area_per_sqm,
        'latest_land_value_mixed_area_per_sqm', latest_land_value_mixed_area_per_sqm,
        'latest_land_value_others_per_sqm', latest_land_value_others_per_sqm,
        'latest_land_value_residential_area_per_sqm', latest_land_value_residential_area_per_sqm,
        'latest_mean_net_rent_sqm', latest_mean_net_rent_sqm,
        'latest_land_value_sqm', latest_land_value_sqm,
        'protection_zones_count', protection_zones_count,
        'total_area_em_ha', total_area_em_ha,
        'total_area_es_ha', total_area_es_ha,
        'latest_mean_net_rent_sqm', latest_mean_net_rent_sqm,
        'latest_land_value_commercial_area_per_sqm', latest_land_value_commercial_area_per_sqm
    ) AS real_estate_value_json,

    -- LONG-TERM LISTINGS JSON
    jsonb_build_object(
        'long_term_listings_count', long_term_listings_count,
        'avg_long_term_listings_price', avg_long_term_listings_price,
        'median_long_term_listings_price', median_long_term_listings_price,
        'avg_long_term_listings_rooms', avg_long_term_listings_rooms,
        'avg_long_term_listings_surface_sqm', avg_long_term_listings_surface_sqm,
        'avg_price_rooms', jsonb_build_object(
            '1', avg_price_rooms_1,
            '1_5', avg_price_rooms_1_5,
            '2', avg_price_rooms_2,
            '2_5', avg_price_rooms_2_5,
            '3', avg_price_rooms_3,
            '3_5', avg_price_rooms_3_5,
            '4', avg_price_rooms_4,
            '4_5', avg_price_rooms_4_5,
            '5', avg_price_rooms_5,
            '5_5', avg_price_rooms_5_5,
            '6', avg_price_rooms_6,
            '7', avg_price_rooms_7,
            '7_5', avg_price_rooms_7_5,
            '8', avg_price_rooms_8,
            '9', avg_price_rooms_9
        )
    ) AS long_term_listings_json,

    -- SHORT-TERM LISTINGS / TOURISM JSON
    jsonb_build_object(
        'short_term_listings_count', short_term_listings_count,
        'avg_short_term_rating', avg_short_term_rating,
        'entire_home_apt_count', entire_home_apt_count,
        'hotel_room_count', hotel_room_count,
        'private_room_count', private_room_count,
        'shared_room_count', shared_room_count,
        'avg_price_entire_home_apt', avg_price_entire_home_apt,
        'avg_price_hotel_room', avg_price_hotel_room,
        'avg_price_private_room', avg_price_private_room,
        'avg_price_shared_room', avg_price_shared_room,
        'tourist_coefficient', tourist_coefficient
    ) AS short_term_listings_json,

    -- OTHER DETAILS (catch-all)
    jsonb_build_object(
        'venues_per_10000_inhabitants', venues_per_10000_inhabitants,
        'banks_count', banks_count,
        'atms_count', atms_count,
        'banks_wheelchair_accessible_ratio', banks_wheelchair_accessible_ratio,
        'wheelchair_venues_yes_count', wheelchair_venues_yes_count,
        'wheelchair_venues_no_count', wheelchair_venues_no_count,
        'wheelchair_venues_limited_count', wheelchair_venues_limited_count,
        'startchancen_flags_count', startchancen_flags_count,
        'students_per_teacher_ratio', students_per_teacher_ratio,
        'universities_count', universities_count,
        'university_students_count', university_students_count,
        'latest_mean_net_rent_sqm', latest_mean_net_rent_sqm
    ) AS other_details_json

FROM berlin_source_data.district_level_aggregated AS dla;


    
"""
with engine.begin() as conn:
    conn.execute(text(insert_query))

### Show full table

In [12]:
query = f"""
SELECT *
FROM district_details

"""

# Execute the query
with engine.connect() as conn:
    df= pd.read_sql(text(query), conn)
    conn.commit()  # commit the transaction
df

,district_id,room_distribution_json,school_stats_json,pool_stats_json,crime_totals_json,crime_violent_json,crime_property_json,crime_drugs_json,crime_other_json,demographics_json,healthcare_json,transport_json,parks_recreation_json,land_use_json,ownership_json,real_estate_value_json,long_term_listings_json,short_term_listings_json,other_details_json
0,11001001,"{'counts': {'rooms_1': 39, 'rooms_2': 59, 'roo...","{'schools_count': 87, 'students_count': 34418,...","{'pools_count': 9, 'freibad_count': 2, 'hotelb...","{'crime_weighted_score': 45, 'crime_cases_tota...","{'robbery_cases_count': 0, 'assault_total_case...","{'theft_total_cases_count': 0, 'vehicle_theft_...","{'drug_crimes_cases_count': 0, 'drug_offense_c...","{'arson_cases_count': 10, 'arson_total_cases_c...","{'germans_population_pct': 62.73, 'latest_popu...","{'hospitals_count': 10, 'hospital_beds_count':...","{'ubahn_lines_count': 9, 'bus_tram_stops_count...","{'bars_count': 236, 'cafes_count': 540, 'parks...","{'latest_total_area_ha': 3940, 'latest_water_a...","{'no_ownership_count': 0, 'public_ownership_co...","{'total_area_em_ha': 686.6, 'total_area_es_ha'...","{'avg_price_rooms': {'1': 1120.85, '2': 1794.8...","{'hotel_room_count': 41, 'shared_room_count': ...","{'atms_count': 31, 'banks_count': 48, 'univers..."
1,11002002,"{'counts': {'rooms_1': 16, 'rooms_2': 43, 'roo...","{'schools_count': 73, 'students_count': 33454,...","{'pools_count': 8, 'freibad_count': 2, 'hotelb...","{'crime_weighted_score': 945, 'crime_cases_tot...","{'robbery_cases_count': 0, 'assault_total_case...","{'theft_total_cases_count': 0, 'vehicle_theft_...","{'drug_crimes_cases_count': 0, 'drug_offense_c...","{'arson_cases_count': 0, 'arson_total_cases_co...","{'germans_population_pct': 69.25, 'latest_popu...","{'hospitals_count': 4, 'hospital_beds_count': ...","{'ubahn_lines_count': 6, 'bus_tram_stops_count...","{'bars_count': 167, 'cafes_count': 392, 'parks...","{'latest_total_area_ha': 2040, 'latest_water_a...","{'no_ownership_count': 0, 'public_ownership_co...","{'total_area_em_ha': 981.3, 'total_area_es_ha'...","{'avg_price_rooms': {'1': 1165.44, '2': 1608.1...","{'hotel_room_count': 5, 'shared_room_count': 3...","{'atms_count': 15, 'banks_count': 21, 'univers..."
2,11003003,"{'counts': {'rooms_1': 21, 'rooms_2': 47, 'roo...","{'schools_count': 110, 'students_count': 49106...","{'pools_count': 9, 'freibad_count': 1, 'hotelb...","{'crime_weighted_score': 168, 'crime_cases_tot...","{'robbery_cases_count': 0, 'assault_total_case...","{'theft_total_cases_count': 0, 'vehicle_theft_...","{'drug_crimes_cases_count': 0, 'drug_offense_c...","{'arson_cases_count': 0, 'arson_total_cases_co...","{'germans_population_pct': 80.3, 'latest_popul...","{'hospitals_count': 7, 'hospital_beds_count': ...","{'ubahn_lines_count': 1, 'bus_tram_stops_count...","{'bars_count': 107, 'cafes_count': 282, 'parks...","{'latest_total_area_ha': 10322, 'latest_water_...","{'no_ownership_count': 0, 'public_ownership_co...","{'total_area_em_ha': 833.2, 'total_area_es_ha'...","{'avg_price_rooms': {'1': 1012.71, '2': 1385.3...","{'hotel_room_count': 20, 'shared_room_count': ...","{'atms_count': 19, 'banks_count': 26, 'univers..."
3,11004004,"{'counts': {'rooms_1': 24, 'rooms_2': 44, 'roo...","{'schools_count': 85, 'students_count': 37921,...","{'pools_count': 15, 'freibad_count': 3, 'hotel...","{'crime_weighted_score': 225, 'crime_cases_tot...","{'robbery_cases_count': 0, 'assault_total_case...","{'theft_total_cases_count': 0, 'vehicle_theft_...","{'drug_crimes_cases_count': 0, 'drug_offense_c...","{'arson_cases_count': 0, 'arson_total_cases_co...","{'germans_population_pct': 72.78, 'latest_popu...","{'hospitals_count': 9, 'hospital_beds_count': ...","{'ubahn_lines_count': 6, 'bus_tram_stops_count...","{'bars_count': 76, 'cafes_count': 391, 'parks_...","{'latest_total_area_ha': 6469, 'latest_water_a...","{'no_ownership_count': 0, 'public_ownership_co...","{'total_area_em_ha': 397.2, 'total_area_es_ha'...","{'avg_price_roo

In [13]:
query = f"""
SELECT district_id, (crime_property_json->>'theft_total_cases_count')::bigint AS theft_total
FROM district_details
ORDER BY theft_total DESC
LIMIT 20;
;
"""

# Execute the query
with engine.connect() as conn:
    df= pd.read_sql(text(query), conn)
    conn.commit()  # commit the transaction
df


,district_id,theft_total
0,11001001,0
1,11002002,0
2,11003003,0
3,11004004,0
4,11005005,0
5,11006006,0
6,11007007,0
7,11008008,0
8,11009009,0
9,11010010,0


### district_time_series table

In [14]:
query = f"""
WITH rent_agg AS (
        SELECT district_id, 
        year, 
        AVG(mean_net_rent_per_m2) AS avg_rent,
        AVG(number_of_cases) AS avg_cases
        FROM berlin_source_data.rent_stats_per_neighborhood
        GROUP BY district_id, year
)
SELECT 
    rs.district_id,
    rs.year,
    cs.crime_type_english AS crime_type,
    cs.total_number_cases AS crime_cases,
    cs.frequency_100k,
    rs.avg_rent AS median_net_rent_per_m2,
    rs.avg_cases AS number_of_cases

FROM berlin_source_data.crime_statistics cs
LEFT JOIN rent_agg rs
    ON cs.district_id = rs.district_id
"""

# Execute the query
district_time_series_df = pd.read_sql(text(query), engine)
district_time_series_df

,district_id,year,crime_type,crime_cases,frequency_100k,median_net_rent_per_m2,number_of_cases
0,11001001,2018,Arson,13,34.0,12.94,6716.0
1,11001001,2024,Arson,13,34.0,19.91,4169.0
2,11001001,2013,Arson,13,34.0,9.52,6972.0
3,11001001,2020,Arson,13,34.0,13.85,4669.0
4,11001001,2015,Arson,13,34.0,10.94,6340.0
...,...,...,...,...,...,...,...
369065,11012012,2022,Vehicle Theft,13,NaN,10.53,1635.0
369066,11012012,2024,Vehicle Theft,13,NaN,13.54,1938.0
369067,11012012,2014,Vehicle Theft,13,NaN,7.30,3833.0
369068,11012012,2019,Vehicle Theft,13,NaN,9.75,3046.0
